In [1]:
import sys
sys.path.append('..')


from sympy.ntheory import factorint
from tools import *

import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np
import torchtt as tt

In [2]:
def bfs(meshname, ind_type):
    ds = nc.Dataset(meshname)
    if ind_type == 'cells':
        nodes_on_node = ds.variables['cellsOnCell'][:]
        visited = np.zeros(ds.dimensions['nCells'].size, dtype=int)

        root_ind = 0
    elif ind_type == 'edges':
        nodes_on_node = ds.variables['edgesOnEdge'][:]
        visited = np.zeros(ds.dimensions['nEdges'].size, dtype=int)

        edges_on_cell = ds.variables['edgesOnCell'][:]
        root_ind = edges_on_cell[0, 0] - 1
    else:
        print(f'ind_type of {ind_type} not valid.')
        return
    # END if
    ds.close()

    #root_ind = 0
        
    sorted_data = []
    queue = [(root_ind, 0)]
    visited[root_ind] = 1
    while queue:
        cur_ind, cur_depth = queue.pop(0)
        sorted_data.append((cur_ind, cur_depth))
    
        adjacent_inds = np.array(nodes_on_node[cur_ind][nodes_on_node[cur_ind] != 0]) - 1
        for ind in adjacent_inds:
            if not visited[ind]:
                queue.append((ind, cur_depth + 1))
                visited[ind] = 1
            # END if
        # END for
    # END while   
    ordered_inds = np.array([data[0] for data in sorted_data])
    ordered_levels = np.array([data[1] for data in sorted_data])

    return ordered_inds, ordered_levels
# END bfs()

In [3]:
mesh_num = 9
meshname = f'io/mesh_cvt_{mesh_num}.nc'


ordered_inds, ordered_levels = bfs(meshname, 'cells')
print(ordered_inds.size, ordered_levels.size)

if False:
    ds = nc.Dataset(meshname, 'a', format='NETCDF4')
    bfs_depth_name = 'bfs_depth_cell'
    try:
        ds.createVariable(bfs_depth_name, 'i', ('nCells'))
        print(f'created variable {bfs_depth_name}')
    except:
        print(f'variable {bfs_depth_name} already exists')
    # END try
    ds.variables[bfs_depth_name][ordered_inds] = ordered_levels
    ds.close()
# END if


ordered_inds, ordered_levels = bfs(meshname, 'edges')
print(ordered_inds.size, ordered_levels.size)

if False:
    ds = nc.Dataset(meshname, 'a', format='NETCDF4')
    bfs_depth_name = 'bfs_depth_edge'
    try:
        ds.createVariable(bfs_depth_name, 'i', ('nEdges'))
        print(f'created variable {bfs_depth_name}')
    except:
        print(f'variable {bfs_depth_name} already exists')
    # END try
    ds.variables[bfs_depth_name][ordered_inds] = ordered_levels
    ds.close()
# END if

2621442 2621442
7864320 7864320


In [4]:
def get_node_loc(ind, mesh_data):
    return np.array([mesh_data['xnode'][ind],
                     mesh_data['ynode'][ind],
                     mesh_data['znode'][ind]])
# END get_node_loc()


def kite_side_adjacency(meshname):
    ds = nc.Dataset(meshname)
    nodes_on_node = ds.variables['edgesOnEdge'][:]
    nnodes = ds.dimensions['nEdges'].size
    xnode = ds.variables['xEdge'][:]
    ynode = ds.variables['yEdge'][:]
    znode = ds.variables['zEdge'][:]
    ds.close()
    
    mesh_data = mesh_data  = {'xnode': xnode,
                      'ynode': ynode,
                      'znode': znode,
                      'nodes_on_node': nodes_on_node,
                      'nnodes': nnodes}
    
    kite_side_adj = np.zeros([nnodes, 6], dtype=int)
    for ind in range(nnodes):
        pt = get_node_loc(ind, mesh_data)
        adj_inds = nodes_on_node[ind][np.where(nodes_on_node[ind] != 0)] - 1
    
        dists = np.zeros([adj_inds.size])
        for i, adj_ind in enumerate(adj_inds):
            adj_pt = get_node_loc(adj_ind, mesh_data)
            dists[i] = np.linalg.norm(pt - adj_pt)
        # END for
    
        dist_sorted_inds = np.argsort(dists)
        kite_side_adj[ind, :4] = adj_inds[dist_sorted_inds][:4] + 1
        kite_side_adj[ind, 4:] = 0
    # END for
    return kite_side_adj
# END

In [5]:
kite_side_adj = kite_side_adjacency(meshname)
print(kite_side_adj)

if False:
    ds = nc.Dataset(meshname, 'a', format='NETCDF4')
    var_name = 'kite_side_adjacency'
    try:
        ds.createVariable(var_name, 'i', ('nEdges', 'maxEdges'))
        print(f'created variable {var_name}')
    except:
        print(f'variable {var_name} already exists')
    # END try
    ds.variables[var_name][:, :] = kite_side_adj
    ds.close()
# END if

[[2908768  864915       2 4428696       0       0]
 [2908772       1 4861877 4428696       0       0]
 [5366741  140051 2908781 4236889       0       0]
 ...
 [6154859 7812816 6815530 2386117       0       0]
 [1807691 4775343 1807682 4573158       0       0]
 [ 391047 7595297 7274134 4227839       0       0]]


In [6]:
def get_mesh_data(meshname, ind_type):
    ### get mesh data
    ds = nc.Dataset(meshname)
    radius = ds.__dict__['sphere_radius']
    if ind_type == 'cells':
        xnode = ds.variables['xCell'][:]
        ynode = ds.variables['yCell'][:]
        znode = ds.variables['zCell'][:]
        latnode = ds.variables['latCell'][:]
        nodes_on_node = ds.variables['cellsOnCell'][:]
        node_depth = ds.variables['bfs_depth_cell'][:]
        nnodes = ds.dimensions['nCells'].size

        root_ind = 0
    elif ind_type == 'edges':
        xnode = ds.variables['xEdge'][:]
        ynode = ds.variables['yEdge'][:]
        znode = ds.variables['zEdge'][:]
        latnode = ds.variables['latEdge'][:]
        #nodes_on_node = ds.variables['edgesOnEdge'][:]  # use kite-sides instead?
        nodes_on_node = ds.variables['kite_side_adjacency'][:]
        node_depth = ds.variables['bfs_depth_edge'][:]
        nnodes = ds.dimensions['nEdges'].size

        edges_on_edge = ds.variables['edgesOnEdge'][:]

        cells_on_edge = ds.variables['cellsOnEdge'][:]
        edges_on_cell = ds.variables['edgesOnCell'][:]
        root_ind = edges_on_cell[0, 0] - 1
    else:
        print(f'ind_type of {ind_type} not valid.')
        return
    # END if
    ds.close()

    ### build meta dict for mesh data
    mesh_data  = {'xnode': xnode,
                  'ynode': ynode,
                  'znode': znode,
                  'latnode': latnode,
                  'nodes_on_node': nodes_on_node,
                  'node_depth': node_depth,
                  'nnodes': nnodes,
                  'radius': radius}
    if ind_type == 'edges':
        mesh_data.update({'cells_on_edge': cells_on_edge,
                          'edges_on_cell': edges_on_cell,
                          'edges_on_edge': edges_on_edge})
    # END if

    return mesh_data, root_ind
# END get_mesh_data()


def order_by_angle(root_ind, root_vec, adjacent, mesh_data):
    root_pt = get_node_loc(root_ind, mesh_data)
    
    angles = np.zeros(adjacent.size)
    for i, ind in enumerate(adjacent):
        pt = get_node_loc(ind, mesh_data)
        vec = pt - root_pt
        
        angle = np.arccos(np.dot(root_vec, vec) / np.linalg.norm(vec))
        if vec[1] < 0:
            angle = 2 * np.pi - angle
        # END if
        angles[i] = angle
    # END for
    
    sorted_inds = np.argsort(angles)
    return adjacent[sorted_inds], angles[sorted_inds]
# END order_by_angle


def dist_from_origin(vec, pt, mesh_data):
    origin = np.array([0, 0, 0])
    radius = mesh_data['radius']
    
    new_pt = pt + (radius / 10) * vec
    return np.linalg.norm(origin - new_pt)
# END dist_from_origin()


def get_next_in_spiral(prev_ind, cur_ind, candidate_inds, mesh_data):
    radius = mesh_data['radius']
    latnode = mesh_data['latnode']
    node_depth = mesh_data['node_depth']
    
    prev_pt = get_node_loc(prev_ind, mesh_data)
    cur_pt = get_node_loc(cur_ind, mesh_data)
    root_vec = cur_pt - prev_pt
    root_vec = root_vec / np.linalg.norm(root_vec)

    dists = np.zeros(len(candidate_inds))
    for i, ind in enumerate(candidate_inds):
        pt = get_node_loc(ind, mesh_data)
        vec = pt - prev_pt
        vec = vec / np.linalg.norm(vec)
        cross = np.cross(root_vec, vec)

        dists[i] = dist_from_origin(cross, pt, mesh_data)
    # END for
    inside_inds = candidate_inds[np.where(dists < radius)]
    inside_dists = dists[np.where(dists < radius)]
    outside_inds = candidate_inds[np.where(dists >= radius)]
    outside_dists = dists[np.where(dists >= radius)]

    # only take a cross pointing outside of the sphere if
    # there are none pointing inside available
    if inside_inds.size > 0:
        # get ind for cross closest to 1
        ind = inside_inds[np.argmin(np.abs(1 - inside_dists))]
    else:
        # get ind for cross closest to 1
        ind = outside_inds[np.argmin(np.abs(1 - outside_dists))]
    # END if

    return ind
# END get_next_in_spiral()


def borders_prev_depth(cur_ind, mesh_data):
    nodes_on_node = mesh_data['nodes_on_node']
    node_depth = mesh_data['node_depth']

    cur_depth = node_depth[cur_ind]
    prev_depth = cur_depth - 1

    adjacent_inds = np.array(nodes_on_node[cur_ind][nodes_on_node[cur_ind] != 0]) - 1
    for ind in adjacent_inds:
        if node_depth[ind] == prev_depth:
            return True
        # END if
    # END for
    return False
# END boarders_prev_depth()


def edge_on_pentagon(cur_ind, mesh_data):
    cells_on_edge = mesh_data['cells_on_edge']
    edges_on_cell = mesh_data['edges_on_cell']

    cells_on_ind = cells_on_edge[cur_ind][cells_on_edge[cur_ind] != 0] - 1
    for cell in cells_on_ind:
        edges_on_cur_cell = edges_on_cell[cell]
        nedges_on_cell = edges_on_cur_cell[edges_on_cur_cell != 0].size
        if nedges_on_cell == 5:
            return True
        # END if
    # END for
    return False
# END edge_on_petagram()


def pentaproblem(cur_ind, mesh_data):
    node_depth = mesh_data['node_depth']
    nodes_on_node = mesh_data['nodes_on_node']
    latnode = mesh_data['latnode']

    cur_depth = node_depth[cur_ind]
    
    adj_inds = nodes_on_node[cur_ind][nodes_on_node[cur_ind] != 0] - 1
    adj_inds = adj_inds[node_depth[adj_inds] == cur_depth]
    for ind in adj_inds:
        if not borders_prev_depth(ind, mesh_data):
            adj_adj_inds = nodes_on_node[ind][nodes_on_node[ind] != 0] - 1
            adj_adj_inds = adj_adj_inds[node_depth[adj_adj_inds] == cur_depth]
            penta_inds = []
            npenta = 0
            for iind in adj_adj_inds:
                if edge_on_pentagon(iind, mesh_data) and borders_prev_depth(iind, mesh_data):
                    penta_inds.append(iind)
                    npenta += 1
                # END if
            # END for
            if npenta == 2 and penta_inds[0] in nodes_on_node[penta_inds[1]] - 1:
                # AND check if the two penta edges are adjacent??? (to address the 64 vs 65 problem)
                return ind
            # END if
        # END if
    # END for
    return -1
# END pentaproblem()


def get_prefered_edge(cur_ind, candidate_inds, sorted_inds, mesh_data):
    node_depth = mesh_data['node_depth']
    nodes_on_node = mesh_data['nodes_on_node']
    latnode = mesh_data['latnode']

    # first prefer penta problem,
    # second prefer border,
    # third prefer small into cross

    used_pentaproblem = True
    ind = pentaproblem(cur_ind, mesh_data)
    if ind == -1 or latnode[cur_ind] < 0:
        used_pentaproblem = False
        prev_depth_border_inds = []
        for ind in candidate_inds:
            if borders_prev_depth(ind, mesh_data):
                prev_depth_border_inds.append(ind)
            # END if
        # END for
        prev_depth_border_inds = np.array(prev_depth_border_inds)

        if prev_depth_border_inds.size == 1:
            ind = prev_depth_border_inds[0]
        elif prev_depth_border_inds.size > 1:
            ind = get_next_in_spiral(sorted_inds[-2], cur_ind, prev_depth_border_inds, mesh_data)
        else:
            ind = get_next_in_spiral(sorted_inds[-2], cur_ind, candidate_inds, mesh_data)
        # END if
    # END if

    return ind, candidate_inds[candidate_inds != ind], used_pentaproblem
# END get_prefered_edge()

In [7]:
ds = nc.Dataset(meshname)
nedges = ds.dimensions['nEdges'].size
cells_on_edge = ds.variables['cellsOnEdge'][:]
edges_on_cell = ds.variables['edgesOnCell'][:]
xnode = ds.variables['xEdge'][:]
ynode = ds.variables['yEdge'][:]
znode = ds.variables['zEdge'][:]
latnode = ds.variables['latEdge'][:]
nodes_on_node = ds.variables['kite_side_adjacency'][:]
node_depth = ds.variables['bfs_depth_edge'][:]
nnodes = ds.dimensions['nEdges'].size
ds.close()

mesh_data = {'xnode': xnode,
             'ynode': ynode,
             'znode': znode,
             'latnode': latnode,
             'nodes_on_node': nodes_on_node,
             'node_depth': node_depth,
             'nnodes': nnodes,
             'cells_on_edge': cells_on_edge,
             'edges_on_cell': edges_on_cell}

#nedges_on_pentagons = 0
#edges_on_pentagons = []
#pentaproblems = []
#for ind in range(nedges):
#    num = pentaproblem(ind, mesh_data)
#    if num != -1:
#        pentaproblems.append(ind)
    # END if
    #if edge_on_pentagon(ind, mesh_data):
    #    nedges_on_pentagons += 1
    #    edges_on_pentagons.append(ind)
# END for

#print(np.unique(np.array(pentaproblems)))
#print(nedges_on_pentagons)
#print(edges_on_pentagons)

#print(pentaproblem(198642-1, mesh_data))

In [8]:
def nightmare(sorted_inds, visited, candidate_inds, mesh_data):
    node_depth = mesh_data['node_depth']
    
    cur_ind = sorted_inds[-1]
    cur_depth = node_depth[cur_ind]

    ind = pentaproblem(cur_ind, mesh_data)
    if ind != -1 and not visited[ind]:
        sorted_inds.append(ind)
        visited[ind] = 1
        other_ind = candidate_inds[candidate_inds != ind][0]

        cur_ind = sorted_inds[-1]
        cur_depth = node_depth[cur_ind]

        adjacent_inds = np.array(nodes_on_node[cur_ind][nodes_on_node[cur_ind] != 0]) - 1
        candidate_inds = adjacent_inds[node_depth[adjacent_inds] == cur_depth]
        candidate_inds = candidate_inds[visited[candidate_inds] == 0]
        candidate_inds = candidate_inds[candidate_inds != other_ind]

        ind = get_next_in_spiral(sorted_inds[-2], cur_ind, candidate_inds, mesh_data)
        sorted_inds.append(ind)
        visited[ind] = 1

        ind = candidate_inds[candidate_inds != ind][0]
        sorted_inds.append(ind)
        visited[ind] = 1

        ind = other_ind
        sorted_inds.append(ind)
        visited[ind] = 1  
    else:
        prev_depth_border_inds = []
        for ind in candidate_inds:
            if borders_prev_depth(ind, mesh_data):
                prev_depth_border_inds.append(ind)
            # END if
        # END for
        prev_depth_border_inds = np.array(prev_depth_border_inds)
        
        if prev_depth_border_inds.size == 1:
            ind = prev_depth_border_inds[0]
        elif prev_depth_border_inds.size > 1:
            ind = get_next_in_spiral(sorted_inds[-2], cur_ind, prev_depth_border_inds, mesh_data)
        else:
            ind = get_next_in_spiral(sorted_inds[-2], cur_ind, candidate_inds, mesh_data)
        # END if
        
        sorted_inds.append(ind)
        visited[ind] = 1
    # END if

    return sorted_inds, visited
# END nightmare()

In [9]:
def spiral(meshname, ind_type='cells'):
    mesh_data, root_ind = get_mesh_data(meshname, ind_type)

    nodes_on_node = mesh_data['nodes_on_node']
    node_depth = mesh_data['node_depth']
    nnodes = mesh_data['nnodes']
    if ind_type == 'edges':
        edges_on_edge = mesh_data['edges_on_edge']
    # END if

    ### step 0: init
    sorted_inds = [root_ind]
    visited = np.zeros(nnodes, dtype=int)
    visited[root_ind] = 1

    ### step 1: manually get second node
    root_vec = np.array([1, 0, 0])
    adjacent_inds = np.array(nodes_on_node[root_ind][nodes_on_node[root_ind] != 0]) - 1
    adjacent_inds, angles = order_by_angle(root_ind, root_vec, adjacent_inds, mesh_data)

    sorted_inds.append(adjacent_inds[0])
    visited[adjacent_inds[0]] = 1

    while not np.all(visited):
        ### step 2: choose between two candidate cells within current depth
        cur_ind = sorted_inds[-1]
        cur_depth = node_depth[cur_ind]
        
        adjacent_inds = np.array(nodes_on_node[cur_ind][nodes_on_node[cur_ind] != 0]) - 1
        candidate_inds = adjacent_inds[node_depth[adjacent_inds] == cur_depth]

        ind = get_next_in_spiral(sorted_inds[-2], cur_ind, candidate_inds, mesh_data)
        sorted_inds.append(ind)
        visited[ind] = 1
    
        ### step 3: add all remaining cells at current depth
        cur_depth_inds = np.where(node_depth == cur_depth)[0]
        #print(cur_depth)
        while not np.all(visited[cur_depth_inds]):
            cur_ind = sorted_inds[-1]
            cur_depth = node_depth[cur_ind]
            # first get all adjacent inds
            adjacent_inds = np.array(nodes_on_node[cur_ind][nodes_on_node[cur_ind] != 0]) - 1
            # then, remove all not from the current depth
            candidate_inds = adjacent_inds[node_depth[adjacent_inds] == cur_depth]
            # then, only allow an ind that hasn't been visited
            candidate_inds = candidate_inds[visited[candidate_inds] == 0]
            if candidate_inds.size > 1:
                sorted_inds, visited = nightmare(sorted_inds, visited, candidate_inds, mesh_data)
            else:
                ind = candidate_inds[0]
                sorted_inds.append(ind)
                visited[ind] = 1
            # END if
        # END all
    
        ### step 4: move up to next depth level
        cur_ind = sorted_inds[-1]
        new_depth = node_depth[cur_ind] + 1
    
        adjacent_inds = np.array(nodes_on_node[cur_ind][nodes_on_node[cur_ind] != 0]) - 1
        candidate_inds = adjacent_inds[node_depth[adjacent_inds] == new_depth]

        if candidate_inds.size == 0:
            # this can happen if we end on an edge that doesn't have kite-side adjacency with an
            # edge at the next depth level
            adjacent_inds = np.array(edges_on_edge[cur_ind][edges_on_edge[cur_ind] != 0]) - 1
            candidate_inds = adjacent_inds[node_depth[adjacent_inds] == new_depth]
        # END if
        ind = get_next_in_spiral(sorted_inds[-2], cur_ind, candidate_inds, mesh_data)    
        sorted_inds.append(ind)
        visited[ind] = 1
    # END while

    return np.array(sorted_inds)
# END spiral()

In [13]:
ind_type = 'edges'
sorted_inds = spiral(meshname, ind_type=ind_type)

print(sorted_inds.size)
#print(sorted_inds)

7864320


In [14]:
if True and ind_type == 'cells':
    ds = nc.Dataset(meshname, 'a', format='NETCDF4')
    spiral_cell_name = 'spiral_order_cell'
    try:
        ds.createVariable(spiral_cell_name, 'i', ('nCells'))
        print(f'created variable {spiral_cell_name}')
    except:
        print(f'variable {spiral_cell_name} already exists')
    # END try
    ds.variables[spiral_cell_name][sorted_inds] = np.arange(ds.dimensions['nCells'].size)
    ds.close()
elif True and ind_type == 'edges':
    ds = nc.Dataset(meshname, 'a', format='NETCDF4')
    spiral_cell_name = 'spiral_order_edge'
    try:
        ds.createVariable(spiral_cell_name, 'i', ('nEdges'))
        print(f'created variable {spiral_cell_name}')
    except:
        print(f'variable {spiral_cell_name} already exists')
    # END try
    ds.variables[spiral_cell_name][sorted_inds] = np.arange(ds.dimensions['nEdges'].size)
    ds.close()
# END if

created variable spiral_order_edge


In [12]:
# this showed that the new alg for spiral sorting the cells 
# gave the same answer as the old alg
# but, this isn't valid anymore since we have overwritten 
# 'spiral_order_cell' with the sorting from the new alg

#ds = nc.Dataset(meshname)
#spiral_inds_cell = np.argsort(ds.variables['spiral_order_cell'])
#ds.close()

#print(np.where(sorted_inds == spiral_inds_cell))
#print(np.all(sorted_inds == spiral_inds_cell))